In [5]:
import ast
import csv
import json
import logging
import math
import os
import random
import time
import tracemalloc
import zipfile

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset


logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

train_path = '/bohr/train-wfpt/v1/'


KEY_COLUMNS = ["index", "task_index", "frame_index", "timestamp"]
OBSERVATION_COLUMN = "observation_state"
SIMULATION_COLUMN = "simulation_positions"
OUTPUT_COLUMNS = KEY_COLUMNS + [OBSERVATION_COLUMN]

RANDOM_SEED = 2026
EPOCHS = 16
BATCH_SIZE = 2048
HIDDEN_SIZE = 256
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 2e-4

TRAIN_FILE = os.path.join(train_path, "train.csv")



def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def parse_vector(value):
    vector = ast.literal_eval(str(value).strip())
    if not isinstance(vector, (list, tuple)) or not vector:
        raise ValueError(f"Invalid vector value: {value}")
    return np.asarray([float(item) for item in vector], dtype=np.float32)


def read_rows(file_path):
    with open(file_path, newline="", encoding="utf-8-sig") as csv_file:
        reader = csv.DictReader(csv_file)
        if reader.fieldnames is None:
            raise ValueError(f"No header found in {file_path}")
        return list(reader)


def make_features(row, simulation=None):
    if simulation is None:
        simulation = parse_vector(row[SIMULATION_COLUMN])
    return np.r_[
        simulation,
        float(row["frame_index"]) / 1200.0,
        float(row["timestamp"]) / 40.0,
    ].astype(np.float32)


def build_train_arrays(train_file):
    features = []
    targets = []
    for row in read_rows(train_file):
        simulation = parse_vector(row[SIMULATION_COLUMN])
        observation = parse_vector(row[OBSERVATION_COLUMN]) - simulation
        features.append(make_features(row, simulation))
        targets.append(observation)
    return np.asarray(features, dtype=np.float32), np.asarray(targets, dtype=np.float32)


def fit_normalizer(values):
    mean = values.mean(axis=0)
    std = values.std(axis=0)
    std[std < 1e-6] = 1.0
    return mean.astype(np.float32), std.astype(np.float32)


class DirectMLP(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(input_dim),
            nn.Linear(input_dim, HIDDEN_SIZE),
            nn.GELU(),
            nn.Linear(HIDDEN_SIZE, HIDDEN_SIZE),
            nn.GELU(),
            nn.Linear(HIDDEN_SIZE, output_dim),
        )

    def forward(self, features):
        return self.net(features)


def train_model(features, targets):
    feature_mean, feature_std = fit_normalizer(features)
    target_mean, target_std = fit_normalizer(targets)
    train_x = (features - feature_mean) / feature_std
    train_y = (targets - target_mean) / target_std

    dataset = TensorDataset(torch.from_numpy(train_x), torch.from_numpy(train_y))
    generator = torch.Generator().manual_seed(RANDOM_SEED)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, generator=generator)

    model = DirectMLP(train_x.shape[1], train_y.shape[1])
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

    history = []
    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss = 0.0
        total_rows = 0
        epoch_start = time.perf_counter()
        for batch_x, batch_y in loader:
            optimizer.zero_grad(set_to_none=True)
            prediction = model(batch_x)
            loss = (prediction - batch_y).pow(2).mean()
            loss.backward()
            optimizer.step()
            total_loss += float(loss.item()) * len(batch_x)
            total_rows += len(batch_x)

        rmse = math.sqrt(total_loss / max(total_rows, 1))
        history.append(
            {
                "epoch": epoch,
                "normalized_target_rmse": rmse,
                "seconds": time.perf_counter() - epoch_start,
            }
        )
        logging.info("Epoch %d/%d - normalized target RMSE %.6f", epoch, EPOCHS, rmse)

    normalizer = {
        "feature_mean": feature_mean,
        "feature_std": feature_std,
        "target_mean": target_mean,
        "target_std": target_std,
    }
    return model, normalizer, history


def format_number(value):
    if abs(value) < 1e-12:
        value = 0.0
    return f"{float(value):.10f}".rstrip("0").rstrip(".")


def format_vector(vector):
    return "[" + ",".join(format_number(item) for item in vector) + "]"


def predict_split(model, normalizer, input_file, output_file):
    rows = read_rows(input_file)
    predictions = []
    model.eval()

    with torch.no_grad():
        for row in rows:
            if str(row[OBSERVATION_COLUMN]).strip() != "":
                continue
            simulation = parse_vector(row[SIMULATION_COLUMN])
            features = make_features(row, simulation)
            features = (features - normalizer["feature_mean"]) / normalizer["feature_std"]
            model_output = model(torch.from_numpy(features).unsqueeze(0)).squeeze(0).numpy()
            model_output = model_output * normalizer["target_std"] + normalizer["target_mean"]
            prediction = simulation + model_output
            predictions.append(
                {
                    "index": row["index"],
                    "task_index": row["task_index"],
                    "frame_index": row["frame_index"],
                    "timestamp": row["timestamp"],
                    "observation_state": format_vector(prediction),
                }
            )

    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    with open(output_file, "w", newline="", encoding="utf-8") as csv_file:
        writer = csv.DictWriter(csv_file, fieldnames=OUTPUT_COLUMNS)
        writer.writeheader()
        writer.writerows(predictions)
    logging.info("Saved %d predictions to %s", len(predictions), output_file)







start = time.perf_counter()
tracemalloc.start()
set_seed(RANDOM_SEED)
torch.set_num_threads(max(1, min(8, os.cpu_count() or 1)))
logging.info("Training direct MLP baseline")
train_features, train_targets = build_train_arrays(TRAIN_FILE)
model, normalizer, history = train_model(train_features, train_targets)




In [6]:
if os.environ.get('DATA_PATH'):
    val_path = os.environ.get("DATA_PATH") + "/"  
else:
    print("Baseline运行时，因为无法读取测试集，所以会有此条报错，属于正常现象")  
    print("When baseline is running, this error message will appear because the test set cannot be read, which is a normal phenomenon.")


VAL_FILE   = os.path.join(val_path,   "val.csv")
TEST_FILE  = os.path.join(val_path,   "test.csv")


SUBMISSION_DIR = './'


predict_split(model, normalizer, VAL_FILE, os.path.join(SUBMISSION_DIR, "submission_val.csv"))
predict_split(model, normalizer, TEST_FILE, os.path.join(SUBMISSION_DIR, "submission_test.csv"))

_, peak = tracemalloc.get_traced_memory()
total_seconds = time.perf_counter() - start



In [7]:
def zip_submission():
    zip_path = "submission.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write("submission_val.csv")
        zf.write("submission_test.csv")
    logging.info("Packed %s (%d bytes)", zip_path, os.path.getsize(zip_path))
zip_submission()
logging.info("Baseline finished in %.2f seconds", total_seconds)